# Анализ текущего уровня потребительской лояльности (NPS) среди клиентов из России одной большой телекоммуникационной компании

<div style="border:solid lightblue 4px; padding: 20px">
 <b> Описание проекта: </b>

    Заказчик этого исследования — большая телекоммуникационная компания, которая оказывает услуги на территории всего СНГ. Перед компанией стоит задача определить текущий уровень потребительской лояльности, или NPS (Net Promoter Score), среди клиентов из России.
    
	Чтобы определить уровень лояльности, клиентам задавали классический вопрос: «Оцените по шкале от 1 до 10 вероятность того, что вы порекомендуете компанию друзьям и знакомым».
    
	Компания провела опрос и попросила вас подготовить дашборд с его итогами. Большую базу данных для такой задачи разворачивать не стали и выгрузили данные в SQLite. 
    
	Чтобы оценить результаты опроса, оценки обычно делят на три группы:
    
* 9-10 баллов — «cторонники» (promoters);
* 7-8 баллов — «нейтралы» (passives);
* 0-6 баллов — «критики» (detractors).


    Итоговое значение NPS рассчитывается по формуле: % «сторонников» - % «критиков».
    

 <b> План проекта: </b>

* Шаг 1. Подключиться к базе данных:
Получить доступ к базе данных (данные выгрузили в SQLite — СУБД, в которой база данных представлена файлом. Для подключения к такой базе достаточно иметь доступ к файлу с расширением .db.
    
    
* Шаг 2. Шаг 2. Выгрузить данные:
    
1) Подготовить данные для дашборда. Преобразовывать данные с помощью Python нельзя — вы должны использовать только SQL-запросы. Нужно собрать в одну витрину данные из разных таблиц. Эту витрину будем использовать для дашборда. 
    
2) Получить следующие данные:
    
1. *user_id*	- Идентификатор клиента;
2. *lt_day*	- Количество дней «жизни» клиента;
3. *is_new*	- Поле хранит информацию о том, является ли клиент новым;
4. *age*	- Возраст;
5. *gender_segment*	- Пол (для удобства работы с полем преобразовали значения в текстовый вид);
6. *os_name*	- Тип операционной системы;
7. *cpe_type_name*	- Тип устройства;
8. *country*	- Страна проживания;
9. *city*	- Город проживания;
10. *age_segment*	- Возрастной сегмент;
11. *traffic_segment*	- Сегмент по объёму потребляемого трафика;
12. *lifetime_segment*	- Сегмент по количеству дней «жизни»;
13. *nps_score*	- Оценка клиента в NPS-опросе;
14. *nps_group*	- Поле хранит информацию о том, к какой группе относится оценка клиента в опросе.

    
* Шаг 3. Создание дашборда в Tableau:
    
Построить дашборд, который представит информацию о текущем уровне NPS среди клиентов и покажет, как этот уровень меняется в зависимости от пользовательских признаков. 
Из дашборда должно быть понятно, какие группы пользователей участвовали в опросе. 
    
* Шаг 4. Ответить на вопросы с помощью дашборда:

1) Как распределены участники опроса по возрасту и полу? Каких пользователей больше: новых или старых? Пользователи из каких городов активнее участвовали в опросе?
    
2) Какие группы пользователей наиболее лояльны к сервису? Какие менее?

3) Какой общий NPS среди всех опрошенных?
    
4) Как можно описать клиентов, которые относятся к группе cторонников (англ. promoters)?
    

* Шаг 5. Оформить презентацию и сделать выводы.

### Шаг 1. Подключение к базе

#### Импортируем библиотеки:

In [1]:
import os
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

#### Формируем путь к БД:

In [2]:
from pathlib import Path

DATA_DIR = Path('data')
path_to_db = DATA_DIR / 'telecomm_csi.db'
path_to_csv = DATA_DIR / 'telecomm_csi_tableau.csv'

engine = None
if path_to_db.exists():
    engine = create_engine(f'sqlite:///{path_to_db}', echo=False)
    print('Подключение к SQLite:', path_to_db)
elif path_to_csv.exists():
    print('SQLite-база не найдена, витрина будет прочитана из CSV:', path_to_csv)
else:
    print('Локальных данных нет: SQL ниже сохранён как витрина для Tableau. '
          'Готовый дашборд — в Tableau Public и в файле .twbx.')

#### Формируем первый запрос к БД, а затем создаем датафрейм по данным запроса:

1) Посмотрим, какие данные содержит в себе таблица user:

In [3]:
query = """
SELECT *
FROM user;
"""

In [4]:
user = pd.read_sql(query, engine)
user.head(10)

,user_id,lt_day,age,gender_segment,os_name,cpe_type_name,location_id,age_gr_id,tr_gr_id,lt_gr_id,nps_score
0,A001A2,2320,45.0,1.0,ANDROID,SMARTPHONE,55,5,5,8,10
1,A001WF,2344,53.0,0.0,ANDROID,SMARTPHONE,21,5,5,8,10
2,A003Q7,467,57.0,0.0,ANDROID,SMARTPHONE,28,6,9,6,10
3,A004TB,4190,44.0,1.0,IOS,SMARTPHONE,38,4,4,8,10
4,A004XT,1163,24.0,0.0,ANDROID,SMARTPHONE,39,2,6,8,10
5,A005O0,5501,42.0,1.0,ANDROID,SMARTPHONE,34,4,6,8,6
6,A0061R,1236,45.0,0.0,ANDROID,SMARTPHONE,55,5,7,8,10
7,A009KS,313,35.0,0.0,ANDROID,SMARTPHONE,28,4,14,5,10
8,A00AES,3238,36.0,1.0,ANDROID,SMARTPHONE,41,4,5,8,10
9,A00F70,4479,54.0,1.0,ANDROID,SMARTPHONE,9,5,8,8,9


In [5]:
# В gender_segment вышли значения 0.0 и 1.0. Проверим тип значений:

user.info(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502493 entries, 0 to 502492
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   user_id         502493 non-null  object 
 1   lt_day          502493 non-null  int64  
 2   age             501939 non-null  float64
 3   gender_segment  501192 non-null  float64
 4   os_name         502493 non-null  object 
 5   cpe_type_name   502493 non-null  object 
 6   location_id     502493 non-null  int64  
 7   age_gr_id       502493 non-null  int64  
 8   tr_gr_id        502493 non-null  int64  
 9   lt_gr_id        502493 non-null  int64  
 10  nps_score       502493 non-null  int64  
dtypes: float64(2), int64(6), object(3)
memory usage: 42.2+ MB


In [6]:
# Изучим значения столбца gender_segment:

query = """
SELECT DISTINCT gender_segment
FROM user;
"""

user = pd.read_sql(query, engine)
print(user)

   gender_segment
0             1.0
1             0.0
2             NaN


In [7]:
query = """
SELECT COUNT(*) AS количество_пропусков
FROM user
WHERE gender_segment IS NULL;
"""

user = pd.read_sql(query, engine)
print(user)

   количество_пропусков
0                  1301


In [8]:
# Посмотрим, есть ли в столбце age отрицательные значения и сколько пропусков:

query = """
SELECT DISTINCT age
FROM user
ORDER BY age;
"""

user = pd.read_sql(query, engine)
print(user)

     age
0    NaN
1   10.0
2   11.0
3   12.0
4   13.0
..   ...
75  84.0
76  85.0
77  86.0
78  87.0
79  89.0

[80 rows x 1 columns]


Есть пропуски в столбце. Узнаем, много ли их:

In [9]:
query = """
SELECT COUNT(*) AS количество_пропусков
FROM user
WHERE age IS NULL;
"""

user = pd.read_sql(query, engine)
print(user)

   количество_пропусков
0                   554


Отрицательных значений нет в столбце age. Можно работать с этими данными.

In [10]:
# Проверим значения в столбце nps_score, там должен  быть диапазон от 1 до 10:

query = """
SELECT DISTINCT nps_score
FROM user
ORDER BY nps_score;
"""

user = pd.read_sql(query, engine)
print(user)

   nps_score
0          1
1          2
2          3
3          4
4          5
5          6
6          7
7          8
8          9
9         10


2) Посмотрим содержание таблицы location:

In [11]:
query = """
SELECT *
FROM location;
"""

location = pd.read_sql(query, engine)
location.head(5)

,location_id,city,country
0,1,Архангельск,Россия
1,2,Астрахань,Россия
2,3,Балашиха,Россия
3,4,Барнаул,Россия
4,5,Белгород,Россия


Посмотрим все уникальные значения (города) столбца city:

In [12]:
query = """
SELECT DISTINCT city
FROM location;
"""

location = pd.read_sql(query, engine)
print(location)

           city
0   Архангельск
1     Астрахань
2      Балашиха
3       Барнаул
4      Белгород
..          ...
57    Челябинск
58    Череповец
59         Чита
60       Якутск
61    Ярославль

[62 rows x 1 columns]


In [13]:
for i in range(len(location)):
    print(location.iloc[i, 0])  # Вывод значений из первого (и единственного) столбца

Архангельск
Астрахань
Балашиха
Барнаул
Белгород
Брянск
Владивосток
Владимир
Волгоград
Волжский
Воронеж
Грозный
Екатеринбург
Иваново
Ижевск
Иркутск
Казань
Калининград
Калуга
Кемерово
Киров
Краснодар
Красноярск
Курск
Липецк
Магнитогорск
Махачкала
Москва
НабережныеЧелны
НижнийНовгород
НижнийТагил
Новокузнецк
Новосибирск
Омск
Оренбург
Пенза
Пермь
РостовнаДону
Рязань
Самара
СанктПетербург
Саранск
Саратов
Смоленск
Сочи
Ставрополь
Сургут
Тверь
Тольятти
Томск
Тула
Тюмень
УланУдэ
Ульяновск
Уфа
Хабаровск
Чебоксары
Челябинск
Череповец
Чита
Якутск
Ярославль


Названия некоторых городов нужно будет скорректировать (НабережныеЧелны в Набережные Челны, НижнийНовгород в Нижний Новгород, НижнийТагил в Нижний Тагил, РостовнаДону в Ростов-на-Дону, СанктПетербург в Санкт-Петербург, УланУдэ в Улан-Удэ). Сделаем это далее через sql-запрос.

3) Cодержание таблицы age_segment:

In [14]:
query = """
SELECT *
FROM age_segment;
"""

age_segment = pd.read_sql(query, engine)
age_segment.head(10)

,age_gr_id,bucket_min,bucket_max,title
0,1,NaN,15.0,01 до 16
1,2,16.0,24.0,02 16-24
2,3,25.0,34.0,03 25-34
3,4,35.0,44.0,04 35-44
4,5,45.0,54.0,05 45-54
5,6,55.0,64.0,06 55-64
6,7,66.0,NaN,07 66 +
7,8,NaN,NaN,08 n/a


Изменим ниже в sql-запросе вид значений в столбце title, используя функцию SUBSTR, которая возвращает часть  строки, начиная с указанного символа.

4) Cодержание таблицы traffic_segment:

In [15]:
query = """
SELECT *
FROM traffic_segment;
"""

traffic_segment = pd.read_sql(query, engine)
traffic_segment.head(30)

,tr_gr_id,bucket_min,bucket_max,title
0,1,0.00,0.00,01 0
1,2,0.00,0.01,01 0-0.01
2,3,0.01,0.10,02 0.01-0.1
3,4,0.10,1.00,03 0.1-1
4,5,1.00,5.00,04 1-5
5,6,5.00,10.00,05 5-10
6,7,10.00,15.00,06 10-15
7,8,15.00,20.00,07 15-20
8,9,20.00,25.00,08 20-25
9,10,25.00,30.00,09 25-30


5) Cодержание таблицы lifetime_segment:

In [16]:
query = """
SELECT *
FROM lifetime_segment;
"""

lifetime_segment = pd.read_sql(query, engine)
lifetime_segment.head(10)

,lt_gr_id,bucket_min,bucket_max,title
0,1,1.0,1.0,01 1
1,2,2.0,2.0,02 2
2,3,3.0,3.0,03 3
3,4,4.0,6.0,04 4-6
4,5,7.0,12.0,05 7-12
5,6,13.0,24.0,06 13-24
6,7,25.0,36.0,07 25-36
7,8,36.0,NaN,08 36+


### Шаг 2. Выгрузка данных

#### Готовим данные для дашборда:

соберем в одну витрину данные из разных таблиц, которые затем будем использовать для дашборда.

In [17]:
query = """
SELECT user_id,

       lt_day,
       CASE
           WHEN lt_day > 365 THEN 'Старый'
           WHEN lt_day <= 365 THEN 'Новый'
       END AS is_new,

       age,
       CASE
           WHEN gender_segment = 0 THEN 'Мужчина'
           WHEN gender_segment = 1 THEN 'Женщина'
           ELSE 'Другое'
       END AS gender_segment,
       
       os_name,
       
       cpe_type_name,
       
       
       l.country,
       
       CASE
           WHEN l.city = 'НабережныеЧелны' THEN 'Набережные Челны'
           WHEN l.city = 'НижнийНовгород' THEN 'Нижний Новгород'
           WHEN l.city = 'НижнийТагил' THEN 'Нижний Тагил'
           WHEN l.city = 'РостовнаДону' THEN 'Ростов-на-Дону'
           WHEN l.city = 'СанктПетербург' THEN 'Санкт-Петербург'
           WHEN l.city = 'УланУдэ' THEN 'Улан-Удэ'
           ELSE l.city 
       END AS city,
       
       
       SUBSTR(age_seg.title, 4) AS age_segment,
       
       
       SUBSTR(traffic_seg.title, 4) AS traffic_segment,
       
       
       SUBSTR(lifetime_seg.title, 4) AS lifetime_segment,
       
       
       nps_score,
       CASE
           WHEN nps_score >= 9 THEN 'Сторонник'
           WHEN nps_score > 6 AND nps_score < 9 THEN 'Нейтрал'
           ELSE 'Критик'
       END AS nps_group
       
       
FROM user AS u
INNER JOIN location as l ON u.location_id = l.location_id
JOIN age_segment AS age_seg ON u.age_gr_id = age_seg.age_gr_id
JOIN traffic_segment AS traffic_seg ON u.tr_gr_id = traffic_seg.tr_gr_id
JOIN lifetime_segment AS lifetime_seg ON u.lt_gr_id = lifetime_seg.lt_gr_id

WHERE age IS NOT NULL;
"""

if engine is not None:
    data = pd.read_sql(query, engine)
elif path_to_csv.exists():
    data = pd.read_csv(path_to_csv)
else:
    data = None
    print('Пропуск выполнения SQL: нет ни .db, ни CSV.')
if data is not None:
    display(data.head(20))

,user_id,lt_day,is_new,age,gender_segment,os_name,cpe_type_name,country,city,age_segment,traffic_segment,lifetime_segment,nps_score,nps_group
0,A001A2,2320,Старый,45.0,Женщина,ANDROID,SMARTPHONE,Россия,Уфа,45-54,1-5,36+,10,Сторонник
1,A001WF,2344,Старый,53.0,Мужчина,ANDROID,SMARTPHONE,Россия,Киров,45-54,1-5,36+,10,Сторонник
2,A003Q7,467,Старый,57.0,Мужчина,ANDROID,SMARTPHONE,Россия,Москва,55-64,20-25,13-24,10,Сторонник
3,A004TB,4190,Старый,44.0,Женщина,IOS,SMARTPHONE,Россия,Ростов-на-Дону,35-44,0.1-1,36+,10,Сторонник
4,A004XT,1163,Старый,24.0,Мужчина,ANDROID,SMARTPHONE,Россия,Рязань,16-24,5-10,36+,10,Сторонник
5,A005O0,5501,Старый,42.0,Женщина,ANDROID,SMARTPHONE,Россия,Омск,35-44,5-10,36+,6,Критик
6,A0061R,1236,Старый,45.0,Мужчина,ANDROID,SMARTPHONE,Россия,Уфа,45-54,10-15,36+,10,Сторонник
7,A009KS,313,Новый,35.0,Мужчина,ANDROID,SMARTPHONE,Россия,Москва,35-44,45-50,7-12,10,Сторонник
8,A00AES,3238,Старый,36.0,Женщина,ANDROID,SMARTPHONE,Россия,Санкт-Петербург,35-44,1-5,36+,10,Сторонник
9,A00F70,4479,Старый,54.0,Женщина,ANDROID,SMARTPHONE,Россия,Волгоград,45-54,15-20,36+,9,Сторонник


In [18]:
df = data
display(df[df['lt_day'].isin([364, 365, 366])][['lt_day', 'is_new']].head())

,lt_day,is_new
205,364,Новый
352,366,Старый
1766,365,Новый
1918,364,Новый
3632,366,Старый


Посмотрим на информацию по получившемуся датафрейму:

In [19]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 501939 entries, 0 to 501938
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   user_id           501939 non-null  object 
 1   lt_day            501939 non-null  int64  
 2   is_new            501939 non-null  object 
 3   age               501939 non-null  float64
 4   gender_segment    501939 non-null  object 
 5   os_name           501939 non-null  object 
 6   cpe_type_name     501939 non-null  object 
 7   country           501939 non-null  object 
 8   city              501939 non-null  object 
 9   age_segment       501939 non-null  object 
 10  traffic_segment   501939 non-null  object 
 11  lifetime_segment  501939 non-null  object 
 12  nps_score         501939 non-null  int64  
 13  nps_group         501939 non-null  object 
dtypes: float64(1), int64(2), object(11)
memory usage: 53.6+ MB


In [20]:
data['gender_segment'].unique()

array(['Женщина', 'Мужчина', 'Другое'], dtype=object)

In [21]:
data['age'].unique()

array([45., 53., 57., 44., 24., 42., 35., 36., 54., 39., 21., 27., 60.,
       34., 47., 37., 43., 33., 31., 25., 51., 28., 41., 40., 46., 48.,
       32., 30., 52., 59., 26., 50., 62., 29., 55., 22., 38., 56., 23.,
       49., 66., 74., 75., 17., 65., 64., 69., 58., 20., 19., 80., 70.,
       81., 63., 67., 68., 72., 15., 79., 18., 73., 14., 71., 61., 16.,
       77., 13., 76., 10., 78., 12., 82., 11., 83., 89., 84., 85., 87.,
       86.])

**ИТОГО** мы имеем 14 столбцов датафрейма, который будем использовать далее для построения дашборда:

1. *user_id*	- Идентификатор клиента;
2. *lt_day*	- Количество дней «жизни» клиента;
3. *is_new*	- Поле хранит информацию о том, является ли клиент новым;
4. *age*	- Возраст;
5. *gender_segment*	- Пол (для удобства работы с полем преобразовали значения в текстовый вид);
6. *os_name*	- Тип операционной системы;
7. *cpe_type_name*	- Тип устройства;
8. *country*	- Страна проживания;
9. *city*	- Город проживания;
10. *age_segment*	- Возрастной сегмент;
11. *traffic_segment*	- Сегмент по объёму потребляемого трафика;
12. *lifetime_segment*	- Сегмент по количеству дней «жизни»;
13. *nps_score*	- Оценка клиента в NPS-опросе;
14. *nps_group*	- Поле хранит информацию о том, к какой группе относится оценка клиента в опросе.


**Всего 501,939 строчки.**


#### Сохраняем датафрейм в виде csv-файла:

In [22]:
if data is not None:
    out_csv = DATA_DIR / 'telecomm_csi_tableau.csv'
    DATA_DIR.mkdir(exist_ok=True)
    data.to_csv(out_csv, index=False)
    print('Сохранено:', out_csv)

Витрина для Tableau сохраняется локально в `data/telecomm_csi_tableau.csv` (файл большой, ~63 МБ, в git не включён). Готовый дашборд опубликован в Tableau Public и лежит в репозитории как `.twbx`.

### Шаг 3. Поcтроение дашборда

Ссылка на дашборд в Tableau Public: https://public.tableau.com/views/2__17279900065660/sheet17?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link